# Demeter Design Reference Environment (DRE) v1.0

This notebook defines a **single calculation baseline** for the Demeter settlement case near **16 Psyche**.

It includes:
- adopted physical constants,
- reference orbital and body parameters,
- thermal environment calculations,
- gravity and rotation calculations,
- Earth--Psyche communication delay,
- a compact engineering baseline for further subsystem sizing.

All values are meant to be a **team-wide common reference** for first-pass design calculations.

In [1]:
import math
import pandas as pd

pd.set_option('display.precision', 6)
pd.set_option('display.max_colwidth', 120)

def sci(x, digits=6):
    return f"{x:.{digits}e}"

## 1. Fundamental physical constants

In [2]:
constants = {
    "c_m_per_s": 299_792_458.0,
    "au_m": 149_597_870_700.0,
    "G_m3_kg_s2": 6.67430e-11,
    "sigma_W_m2_K4": 5.670374419e-8,
    "solar_constant_1au_W_m2": 1361.6,
    "mile_m": 1609.344,
}

constants_df = pd.DataFrame([
    ["Speed of light", "c", constants["c_m_per_s"], "m/s"],
    ["Astronomical unit", "au", constants["au_m"], "m"],
    ["Gravitational constant", "G", constants["G_m3_kg_s2"], "m^3 kg^-1 s^-2"],
    ["Stefan-Boltzmann constant", "sigma", constants["sigma_W_m2_K4"], "W m^-2 K^-4"],
    ["Solar constant at 1 au", "S_1au", constants["solar_constant_1au_W_m2"], "W/m^2"],
], columns=["Quantity", "Symbol", "Value", "Units"])

constants_df

,Quantity,Symbol,Value,Units
0,Speed of light,c,2.997925e+08,m/s
1,Astronomical unit,au,1.495979e+11,m
2,Gravitational constant,G,6.674300e-11,m^3 kg^-1 s^-2
3,Stefan-Boltzmann constant,sigma,5.670374e-08,W m^-2 K^-4
4,Solar constant at 1 au,S_1au,1.361600e+03,W/m^2


## 2. Reference Psyche and orbital parameters

These values define the **design reference environment** for all subsequent calculations.

In [3]:
params = {
    # Orbit around the Sun
    "r_peri_au": 2.54,
    "r_mean_au": 2.93,
    "r_ap_au": 3.32,

    # Psyche body model
    "D_eff_km": 226.0,
    "R_eff_m": 113_000.0,
    "mass_kg": 2.41e19,
    "density_kg_m3": 3990.0,
    "rotation_period_h": 4.196,

    # Thermal assumptions
    "albedo_A": 0.15,
    "emissivity_eps": 0.90,

    # Conservative project baseline for robotics / contact mechanics
    "nominal_g_ref_m_s2": 0.08,

    # Earth-Psyche distance envelope used for comms latency
    "earth_min_distance_miles": 186e6,
    "earth_max_distance_miles": 372e6,
}

params_df = pd.DataFrame([
    ["Perihelion distance", "q", params["r_peri_au"], "au"],
    ["Mean working distance", "a", params["r_mean_au"], "au"],
    ["Aphelion distance", "Q", params["r_ap_au"], "au"],
    ["Effective diameter", "D_eff", params["D_eff_km"], "km"],
    ["Effective radius", "R_eff", params["R_eff_m"], "m"],
    ["Reference mass", "M", params["mass_kg"], "kg"],
    ["Bulk density", "rho", params["density_kg_m3"], "kg/m^3"],
    ["Rotation period", "P_rot", params["rotation_period_h"], "h"],
    ["Bond albedo (design assumption)", "A", params["albedo_A"], "-"],
    ["Emissivity (design assumption)", "eps", params["emissivity_eps"], "-"],
    ["Nominal conservative gravity baseline", "g_ref", params["nominal_g_ref_m_s2"], "m/s^2"],
], columns=["Quantity", "Symbol", "Value", "Units"])

params_df

,Quantity,Symbol,Value,Units
0,Perihelion distance,q,2.540000e+00,au
1,Mean working distance,a,2.930000e+00,au
2,Aphelion distance,Q,3.320000e+00,au
3,Effective diameter,D_eff,2.260000e+02,km
4,Effective radius,R_eff,1.130000e+05,m
5,Reference mass,M,2.410000e+19,kg
6,Bulk density,rho,3.990000e+03,kg/m^3
7,Rotation period,P_rot,4.196000e+00,h
8,Bond albedo (design assumption),A,1.500000e-01,-
9,Emissivity (design assumption),eps,9.000000e-01,-


## 3. Derived orbital solar flux and thermal environment

We use the standard relations

\[
$T_S(r)=\frac{S_{1au}}{r^2},\qquad
T_{eq}=\left(\frac{(1-A)S}{4\,\varepsilon\,\sigma}\right)^{1/4},\qquad
T_{ss}=\left(\frac{(1-A)S}{\varepsilon\,\sigma}\right)^{1/4}.$
\]

Here:
- $T_{eq}$ is the idealized equilibrium temperature of a rapidly redistributed body,
- $T_{ss}$ is the subsolar-point temperature,
- $A=0.15$ and $\varepsilon=0.90$ are **engineering assumptions** used consistently in this notebook.


In [4]:
S1 = constants["solar_constant_1au_W_m2"]
sigma = constants["sigma_W_m2_K4"]
A = params["albedo_A"]
eps = params["emissivity_eps"]

thermal_rows = []
for label, r_au in [("Perihelion", params["r_peri_au"]), ("Mean", params["r_mean_au"]), ("Aphelion", params["r_ap_au"])]:
    S = S1 / (r_au ** 2)
    T_eq = (((1 - A) * S) / (4 * eps * sigma)) ** 0.25
    T_ss = (((1 - A) * S) / (eps * sigma)) ** 0.25
    thermal_rows.append([label, r_au, S, T_eq, T_ss])

thermal_df = pd.DataFrame(thermal_rows, columns=[
    "Orbital case", "r (au)", "Solar flux S (W/m^2)", "T_eq (K)", "T_ss (K)"
])
thermal_df

,Orbital case,r (au),Solar flux S (W/m^2),T_eq (K),T_ss (K)
0,Perihelion,2.54,211.048422,172.175637,243.493122
1,Mean,2.93,158.604061,160.307835,226.709514
2,Aphelion,3.32,123.530266,150.598108,212.977887


## 4. Gravity, rotation and escape speed

We use the following relations:

\[$
\mu = G M,
\qquad
\omega = \frac{2\pi}{P_{rot}},
\qquad
 g = \frac{\mu}{r^2},
\qquad
 a_{cf} = \omega^2 r,
\qquad
 g_{eff} = g - a_{cf},
\qquad
 v_{esc}=\sqrt{\frac{2\mu}{r}}.$
\]

The value $g_{eff}$ below is a **spherical-reference estimate** at the effective radius. For mechanism design and mining robotics, the project still adopts a more conservative **nominal baseline** $g_{ref}=0.08\,\mathrm{m/s^2}$.


In [5]:
G = constants["G_m3_kg_s2"]
M = params["mass_kg"]
r = params["R_eff_m"]
P = params["rotation_period_h"] * 3600.0

mu = G * M
omega = 2 * math.pi / P
g_surface = mu / (r ** 2)
a_centrifugal = omega ** 2 * r
g_eff = g_surface - a_centrifugal
v_escape = math.sqrt(2 * mu / r)

mechanics_df = pd.DataFrame([
    ["Standard gravitational parameter", "mu", mu, "m^3/s^2"],
    ["Angular velocity", "omega", omega, "rad/s"],
    ["Surface gravity at R_eff", "g", g_surface, "m/s^2"],
    ["Centrifugal reduction at R_eff", "a_cf", a_centrifugal, "m/s^2"],
    ["Effective gravity at R_eff", "g_eff", g_eff, "m/s^2"],
    ["Escape speed at R_eff", "v_esc", v_escape, "m/s"],
    ["Adopted conservative nominal gravity", "g_ref", params["nominal_g_ref_m_s2"], "m/s^2"],
], columns=["Quantity", "Symbol", "Value", "Units"])

mechanics_df

,Quantity,Symbol,Value,Units
0,Standard gravitational parameter,mu,1.608506e+09,m^3/s^2
1,Angular velocity,omega,4.159507e-04,rad/s
2,Surface gravity at R_eff,g,1.259696e-01,m/s^2
3,Centrifugal reduction at R_eff,a_cf,1.955070e-02,m/s^2
4,Effective gravity at R_eff,g_eff,1.064189e-01,m/s^2
5,Escape speed at R_eff,v_esc,1.687280e+02,m/s
6,Adopted conservative nominal gravity,g_ref,8.000000e-02,m/s^2


## 5. Earth--Psyche one-way communication delay

We use

\[
t_{light}=\frac{d}{c}
\]

with the project envelope based on distances of approximately **186 million miles** to **372 million miles**.


In [6]:
c = constants["c_m_per_s"]
mile_m = constants["mile_m"]

comm_rows = []
for label, miles in [("Minimum", params["earth_min_distance_miles"]), ("Maximum", params["earth_max_distance_miles"])]:
    d_m = miles * mile_m
    t_s = d_m / c
    comm_rows.append([label, miles, d_m / 1e9, t_s / 60.0])

comm_df = pd.DataFrame(comm_rows, columns=[
    "Case", "Distance (miles)", "Distance (10^9 m)", "One-way light time (min)"
])
comm_df

,Case,Distance (miles),Distance (10^9 m),One-way light time (min)
0,Minimum,186000000.0,299.337984,16.641401
1,Maximum,372000000.0,598.675968,33.282801


## 6. Compact project baseline

These are the values most useful to freeze for subsystem-level calculations.

In [7]:
baseline_df = pd.DataFrame([
    ["Working solar distance", params["r_mean_au"], "au"],
    ["Perihelion / aphelion", f"{params['r_peri_au']} / {params['r_ap_au']}", "au"],
    ["Solar flux at working orbit", round(float(thermal_df.loc[thermal_df['Orbital case'] == 'Mean', 'Solar flux S (W/m^2)'].iloc[0]), 3), "W/m^2"],
    ["Effective radius", params["R_eff_m"], "m"],
    ["Reference mass", sci(params["mass_kg"], 3), "kg"],
    ["Reference gravitational parameter", sci(mu, 3), "m^3/s^2"],
    ["Rotation period", params["rotation_period_h"], "h"],
    ["Spherical-reference effective gravity", round(g_eff, 6), "m/s^2"],
    ["Conservative project gravity baseline", params["nominal_g_ref_m_s2"], "m/s^2"],
    ["Escape speed", round(v_escape, 3), "m/s"],
    ["Thermal equilibrium temperature at working orbit", round(float(thermal_df.loc[thermal_df['Orbital case'] == 'Mean', 'T_eq (K)'].iloc[0]), 3), "K"],
    ["Subsolar temperature at working orbit", round(float(thermal_df.loc[thermal_df['Orbital case'] == 'Mean', 'T_ss (K)'].iloc[0]), 3), "K"],
    ["One-way Earth latency envelope", f"{comm_df['One-way light time (min)'].min():.2f}–{comm_df['One-way light time (min)'].max():.2f}", "min"],
    ["External environment", "Hard vacuum", "qualitative"],
], columns=["Baseline quantity", "Value", "Units"])

baseline_df

,Baseline quantity,Value,Units
0,Working solar distance,2.93,au
1,Perihelion / aphelion,2.54 / 3.32,au
2,Solar flux at working orbit,158.604,W/m^2
3,Effective radius,113000.0,m
4,Reference mass,2.410e+19,kg
5,Reference gravitational parameter,1.609e+09,m^3/s^2
6,Rotation period,4.196,h
7,Spherical-reference effective gravity,0.106419,m/s^2
8,Conservative project gravity baseline,0.08,m/s^2
9,Escape speed,168.728,m/s
